## 1) Imports and helpers

In [1]:
# Core imports
import os
import itertools
from datetime import datetime
import pandas as pd
import jax
import jax.numpy as jnp

# Import the framework helpers used by the CLI runner
from framework.registry import ComparisonRegistry
from framework.runner import run_adapter_benchmark
from framework.adapters import setup_bbob_instances

# toml loader (py3.11 has tomllib; otherwise tomli)
try:
    import tomllib
except Exception:
    import tomli as tomllib  # type: ignore
    
# write export CUDA_VISIBLE_DEVICES=0 in the terminal before running this script
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


print(f'JAX: {jax.__version__} | Backend: {jax.default_backend()} | Device: {jax.devices()[0].device_kind}')

JAX: 0.8.0 | Backend: cpu | Device: cpu


## 1b) HLO Extraction Helper
Import the HLO extraction function for compiler analysis.

In [2]:
from framework.runner import extract_hlo_from_adapter

## 2) Experiment configuration (interactive)
Adjust values here instead of passing a CLI config file.

In [ ]:
# Experiment metadata
EXP_NAME = 'notebook_benchmark'
OUTPUT_DIR = 'results/notebook_runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Grid-like configuration (similar to CLI's toml grid)
# Print available algorithms for user reference
print('Available algorithms:', list(ComparisonRegistry._registry.keys()))
grid = {
    'algorithms': ['Standard_GA'],  # must match keys in ComparisonRegistry
    'tasks': ['rastrigin'],
    'dimensions': [20],
    'pop_sizes': [100],
    'unroll_factors': [1, 25],
    'generations': 100,
    'repeats': 1,  # lower default for quick notebook runs
    'seeds': [42],
}

# Optional hyperparams that will be merged with algorithm defaults
hyperparams = {}

print('Configured experiment grid:')
for k, v in grid.items():
    print(f'  {k}: {v}')

Available algorithms: ['Standard_GA']
Configured experiment grid:
  algorithms: ['Standard_GA']
  tasks: ['rastrigin']
  dimensions: [20]
  pop_sizes: [100]
  unroll_factors: [1, 25]
  generations: 100
  repeats: 10
  seeds: [42]


## 3) Build job queue
This creates the tuple list the CLI would iterate over. You can subset or preview it before running.

In [4]:
# Build the job queue (cartesian product)
job_queue = list(itertools.product(
    grid['algorithms'], grid['tasks'], grid['dimensions'], grid['pop_sizes'], grid.get('unroll_factors', [1])
))
print(f'Total configurations: {len(job_queue)}')
# Preview first few
job_queue[:5]

Total configurations: 2


[('Standard_GA', 'rastrigin', 20, 100, 1),
 ('Standard_GA', 'rastrigin', 20, 100, 25)]

## 4) Single-job runner function
Encapsulates the logic from `benchmarks/cli.py` for one (algo,task,dim,pop,unroll) tuple.

In [5]:
def run_job(algo_key, task, dim, pop, unroll, master_seed, generations, repeats, hypers):
    """Run both adapters for the given job and return packaged result dicts.
    Returns a list with two dicts (one per framework as packaged for CSV).
    """
    spec = ComparisonRegistry.get(algo_key)
    # Merge default hypers with provided ones
    merged_hypers = {**spec.default_hypers, **(hypers or {})}

    # Setup problem instances (MalthusJAX / Evosax adapters expect different objects)
    m_eval, e_prob = setup_bbob_instances(task, dim, master_seed)

    # Factories from registry create adapters given hyperparams etc.
    m_adapter = spec.malthus_factory(pop, dim, master_seed, merged_hypers, m_eval)
    e_adapter = spec.evosax_factory(pop, dim, master_seed, merged_hypers, e_prob)

    # Run benchmarks (returns a small result object from runner)
    res_m = run_adapter_benchmark(m_adapter, generations, master_seed, 'MalthusJAX', pop, unroll, repeats)
    res_e = run_adapter_benchmark(e_adapter, generations, master_seed, 'Evosax', pop, unroll, repeats)

    base = {
        'Algorithm': algo_key, 'Task': task, 'Dim': dim, 'Pop_Size': pop,
        'Unroll': unroll, 'Gens': generations
    }

    def package(res):
        return {
            **base,
            'Framework': res.framework,
            'Mean_GPS': getattr(res, 'mean_gps', None),
            'Mean_Time': getattr(res, 'mean_exec_time', None),
             "Std_Time": getattr(res, 'std_exec_time', None),
            'Compile_Time': getattr(res, 'compile_time', None),
            'Best_Fitness': getattr(res, 'best_fitness_final', None),
        }

    return [package(res_m), package(res_e)]

## 5) Execute the queue (interactive run)
Control the number of jobs to run so the notebook stays responsive. Start with a subset for learning.

In [6]:
# Quick controls: run_all=True will run the entire job_queue (careful).
run_all = True
max_jobs = 2  # when run_all=False, only run first `max_jobs` entries

master_seed = grid['seeds'][0] if grid.get('seeds') else 0
generations = grid['generations']
repeats = grid.get('repeats', 30)

results = []
jobs_to_run = job_queue if run_all else job_queue[:max_jobs]

for i, (algo, task, dim, pop, unroll) in enumerate(jobs_to_run, 1):
    print(f'Running job {i}/{len(jobs_to_run)}: Algo={algo}, Task={task}, Dim={dim}, Pop={pop}, Unroll={unroll}')
    try:
        packaged = run_job(algo, task, dim, pop, unroll, master_seed, generations, repeats, hyperparams)
        results.extend(packaged)
    except Exception as e:
        print('ERROR running job:', e)

# Convert to DataFrame
df = pd.DataFrame(results)
df.head()

Running job 1/2: Algo=Standard_GA, Task=rastrigin, Dim=20, Pop=100, Unroll=1
[MalthusJAX] Compiling (Unroll=1)... Done (0.3765s)
[Evosax] Compiling (Unroll=1)... Done (0.2894s)
Running job 2/2: Algo=Standard_GA, Task=rastrigin, Dim=20, Pop=100, Unroll=25
[MalthusJAX] Compiling (Unroll=25)... Done (2.9554s)
[Evosax] Compiling (Unroll=25)... Done (2.9404s)


,Algorithm,Task,Dim,Pop_Size,Unroll,Gens,Framework,Mean_GPS,Mean_Time,Std_Time,Compile_Time,Best_Fitness
0,Standard_GA,rastrigin,20,100,1,100,MalthusJAX,8501.608945,0.011762,0.000263,0.376516,107.143731
1,Standard_GA,rastrigin,20,100,1,100,Evosax,7532.609135,0.013276,0.000426,0.289401,-4.056567
2,Standard_GA,rastrigin,20,100,25,100,MalthusJAX,8361.364250,0.011960,0.000407,2.955398,107.143731
3,Standard_GA,rastrigin,20,100,25,100,Evosax,7552.052717,0.013241,0.000510,2.940362,-4.056567


## 6) Save & inspect results
Save a CSV copy and display summary statistics.

In [7]:
# Save results with timestamp
if not df.empty:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_file = os.path.join(OUTPUT_DIR, f'notebook_benchmark_{ts}.csv')
    df.to_csv(out_file, index=False)
    print('Saved:', out_file)
    display(df.describe(include='all'))
else:
    print('No results to save (df is empty).')

Saved: results/notebook_runs/notebook_benchmark_20251227_200506.csv


,Algorithm,Task,Dim,Pop_Size,Unroll,Gens,Framework,Mean_GPS,Mean_Time,Std_Time,Compile_Time,Best_Fitness
count,4,4,4.0,4.0,4.000000,4.0,4,4.000000,4.000000,4.000000,4.000000,4.000000
unique,1,1,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN
top,Standard_GA,rastrigin,NaN,NaN,NaN,NaN,MalthusJAX,NaN,NaN,NaN,NaN,NaN
freq,4,4,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,20.0,100.0,13.000000,100.0,NaN,7986.908762,0.012560,0.000401,1.640419,51.543582
std,NaN,NaN,0.0,0.0,13.856406,0.0,NaN,516.598207,0.000811,0.000103,1.510157,64.201522
min,NaN,NaN,20.0,100.0,1.000000,100.0,NaN,7532.609135,0.011762,0.000263,0.289401,-4.056567
25%,NaN,NaN,20.0,100.0,1.000000,100.0,NaN,7547.191822,0.011910,0.000371,0.354737,-4.056567
50%,NaN,NaN,20.0,100.0,13.000000,100.0,NaN,7956.708484,0.012601,0.000417,1.658439,51.543582
75%,NaN,NaN,20.0,100.0,25.000000,100.0,NaN,8396.425424,0.013250,0.000447,2.944121,107.143731


## Notes and next steps
- Use `run_all = True` to run the full grid (may be long).
- Adjust `grid` or `hyperparams` cells to explore different setups.
- The notebook mirrors `benchmarks/cli.py` but keeps execution interactive and educational.

## 7) Extract HLO (Compiler IR) - Optional
Use this to analyze the compiled representation without running benchmarks.

In [11]:
# Example: Extract HLO for a specific configuration
algo_key = 'Standard_GA'
task = 'rastrigin'
framework = 'Evosax' # Evosax
dim = 20
pop = 100
unroll = 25
seed = 42

# Setup the adapter
spec = ComparisonRegistry.get(algo_key)
merged_hypers = {**spec.default_hypers, **hyperparams}
m_eval, e_prob = setup_bbob_instances(task, dim, seed)

# Create MalthusJAX adapter
m_adapter = spec.malthus_factory(pop, dim, seed, merged_hypers, m_eval)

# Extract HLO and save to file
hlo_output_path = f'{OUTPUT_DIR}/hlo_{algo_key}_{task}_d{dim}_p{pop}_u{unroll}_{framework}.txt'
hlo_text = extract_hlo_from_adapter(
    m_adapter, 
    num_gens=100, 
    seed=seed,
    framework_name= framework,
    unroll_factor=unroll,
    output_path=hlo_output_path
)

print(f"\nHLO text length: {len(hlo_text)} characters")
print(f"First 500 chars:\n{hlo_text[:500]}")

[Evosax] Compiling (Unroll=25) to extract HLO... Done (3.8066s)
HLO saved to: results/notebook_runs/hlo_Standard_GA_rastrigin_d20_p100_u25_Evosax.txt

HLO text length: 16370463 characters
First 500 chars:
HloModule jit_scan_loop, is_scheduled=true, entry_computation_layout={(f32[100,20]{1,0}, f32[100]{0}, f32[20]{0}, s32[], f32[], /*index=5*/s32[], u32[2]{0}, f32[], f32[], f32[])->(f32[100,20]{1,0}, f32[100]{0}, f32[20]{0}, s32[], f32[], /*index=5*/s32[], u32[2]{0}, f32[], f32[], f32[], /*index=10*/f32[100]{0}, f32[100]{0}, s32[100]{0}, u32[100,2]{1,0})}, allow_spmd_sharding_propagation_to_parameters={false,false,false,true,false,true,true,true,true,true}, allow_spmd_sharding_propagation_to_outpu
